# 05 — Временной анализ, тренд и прогноз до 2050

Главный научный результат проекта.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import tensorflow as tf
import rasterio

from src import config, data_loader, models, metrics, visualization

unet = tf.keras.models.load_model(
    config.MODELS_DIR / 'unet_best.h5',
    custom_objects=models.get_custom_objects(),
)


## Применяем U-Net ко всем годам Sentinel-2

In [ ]:
results = []
config.MASKS_PRED_DIR.mkdir(parents=True, exist_ok=True)

for year in config.YEARS_SENTINEL2:
    img_path = config.DATA_RAW_SENTINEL2 / f'sentinel2_{year}.tif'
    if not img_path.exists():
        print(f'{year}: снимок не найден, пропуск')
        continue

    print(f'Обрабатываем {year}...', end=' ')
    image = data_loader.load_image(img_path)
    prob_map, binary_mask = models.predict_full_image(image, unet)

    transform, crs, shape, pixel_area_m2 = data_loader.read_raster_meta(img_path)
    area_km2 = metrics.pixels_to_area_km2(binary_mask.sum(), pixel_area_m2)
    results.append({'year': year, 'area_km2': area_km2, 'source': 'Sentinel-2 (U-Net)'})
    print(f'Площадь: {area_km2:.2f} км²')

    with rasterio.open(
        config.MASKS_PRED_DIR / f'mask_pred_{year}.tif', 'w',
        driver='GTiff', height=shape[0], width=shape[1],
        count=1, dtype='uint8', crs=crs, transform=transform,
    ) as dst:
        dst.write(binary_mask, 1)

df = pd.DataFrame(results)
print('\n=== РЕЗУЛЬТАТЫ ПО ГОДАМ ===')
print(df.to_string(index=False))
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(config.TABLES_DIR / 'glacier_areas_by_year.csv', index=False)


## Landsat (2000–2013)

Для старых лет каналов B8A/B12 нет — модель U-Net (обученная на 11 каналах
Sentinel-2) неприменима напрямую. Используем NDSI-пороговый метод
(`config.BEST_NDSI_THRESHOLD`, из ноутбука 03) на доступных каналах
B2/B3/B4/B8/B11 как для Landsat-композитов.


In [ ]:
landsat_results = []

for year in config.YEARS_LANDSAT:
    img_path = config.DATA_RAW_LANDSAT / f'landsat_{year}.tif'
    if not img_path.exists():
        print(f'{year}: снимок Landsat не найден, пропуск')
        continue

    image = data_loader.load_image(img_path)
    ndsi = image[:, :, config.BAND_INDEX['NDSI']]
    binary_mask = (ndsi > config.BEST_NDSI_THRESHOLD).astype('uint8')

    _, crs, _, pixel_area_m2 = data_loader.read_raster_meta(img_path)
    area_km2 = metrics.pixels_to_area_km2(binary_mask.sum(), pixel_area_m2)
    landsat_results.append({'year': year, 'area_km2': area_km2, 'source': 'Landsat (NDSI)'})
    print(f'{year}: площадь (NDSI) = {area_km2:.2f} км²')

df_landsat = pd.DataFrame(landsat_results)
df_all = pd.concat([df_landsat, df], ignore_index=True).sort_values('year')
print(df_all.to_string(index=False))
df_all.to_csv(config.TABLES_DIR / 'glacier_areas_all_years.csv', index=False)


## Статистика тренда

In [ ]:
years = df['year'].values
areas = df['area_km2'].values

trend = metrics.trend_analysis(years, areas)
print('Статистика (Sentinel-2 / U-Net период):')
print(f"  Тренд:   {trend['slope_km2_per_year']:.3f} км²/год")
print(f"  R²:      {trend['r_squared']:.3f}")
print(f"  p-value: {trend['p_value']:.4f} {'(значимо!)' if trend['significant'] else '(не значимо)'}")
print(f"  Изменение: {trend['change_km2']:.2f} км² ({trend['change_percent']:.1f}%)")


## Прогноз до 2050

In [ ]:
future_years, predicted, ci_lower, ci_upper, _ = metrics.forecast_to_2050(years, areas)

visualization.plot_trend_forecast(
    years, areas, future_years, predicted, ci_lower, ci_upper,
    title='Изменение площади ледников Заилийского Алатау: 2015–2050\n(U-Net на снимках Sentinel-2)',
    save_path=config.FIGURES_DIR / 'glacier_trend_forecast.png',
)

area_2050 = predicted[-1]
print(f"Прогноз на 2050: {area_2050:.1f} км² "
      f"(95% CI: {ci_lower[-1]:.1f}-{ci_upper[-1]:.1f} км²)")


## WGMS-валидация (Туюксу)

Заполни словарь `wgms_areas_km2` реальными данными из
https://wgms.ch/products_ref_glaciers/tuyuksuyskiy/ (FoG Browser -> Download Data).


In [ ]:
# Пример структуры (значения-плейсхолдеры -- замени реальными из WGMS!)
wgms_areas_km2 = {
    # 2018: 2.32,
    # 2020: 2.30,
    # 2022: 2.28,
}

predicted_areas_km2 = dict(zip(df['year'], df['area_km2']))

if wgms_areas_km2:
    rmse, common_years, diffs = metrics.rmse_against_wgms(predicted_areas_km2, wgms_areas_km2)
    print(f'RMSE против WGMS ({len(common_years)} лет): {rmse:.4f} км²')
    for y in common_years:
        print(f'  {y}: предсказано {predicted_areas_km2[y]:.3f}, WGMS {wgms_areas_km2[y]:.3f}, '
              f'разница {diffs[y]:+.3f}')
else:
    print('Заполни wgms_areas_km2 данными с wgms.ch для валидации.')


## Водоснабжение Алматы (killer-fact)

In [ ]:
area_loss = abs(trend['change_km2'])
water_impact = metrics.ice_volume_loss_to_water_supply(area_loss)

print(f"Потеря площади за период: {area_loss:.2f} км²")
print(f"Эквивалент в водоснабжении Алматы: "
      f"{water_impact['days_of_supply_equivalent']:.0f} дней "
      f"({water_impact['years_of_supply_equivalent']:.2f} года) потребления города")
